# NB 29 -- Packet P-011: Multi-Model Consensus

**Agent OS v3.0 -- Work Packet Card**

| Field | Value |
|---|---|
| **Packet ID** | P-011 |
| **Decision question** | Do other ML algorithms agree with ExtraTrees on our panel rankings? |
| **Fastest falsifier** | Train GradientBoosting, XGBoost, and RandomForest alongside locked ExtraTrees across 20 random splits; compare top-20 rates for panel devices |
| **Success threshold** | All 3 panel devices have consensus >= 3/4 models agreeing on top-20 in >= 80% of splits --> **Confirmed** |
| **Partial threshold** | >= 2 panel devices meet consensus --> **Promising** |
| **Failure threshold** | < 2 panel devices meet consensus --> **Negative** |

**Panel devices:** 850 (MA-only, T80=3400h), 1320 (MA-FA mixed, T80=940h), 119 (FA-Cs, T80=3423h)

**Methodology:** Reproduce P-005 protocol (20 random splits, seeds 0-19, test-set-only ranking, top-20 rate) with 4 model families.

## 1 -- Imports and Data Loading

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split

# Try XGBoost -- fall back gracefully
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
    print("XGBoost available.")
except Exception:
    HAS_XGB = False
    print("XGBoost not available -- proceeding with 3 models.")

# Load data
df = pd.read_csv("perovskite_stability_clean.csv")
print(f"Dataset: {len(df)} devices")

FEATURES = [
    "Perovskite_band_gap", "Pb", "Sn", "I", "Br", "Cl",
    "MA", "FA", "Cs",
    "first_Prvskt_annealing_temperature", "first_Prvskt_thermal_annealing_time",
    "Perovskite_thickness", "Cell_area_measured",
    "JV_default_Voc", "JV_default_Jsc", "JV_default_FF",
]
TARGET = "Stability_PCE_T80"

X = df[FEATURES].copy()
y = np.log1p(df[TARGET].values)

# Report and handle missing values
nan_counts = X.isna().sum()
nan_cols = nan_counts[nan_counts > 0]
if len(nan_cols) > 0:
    print(f"\nNaN counts per feature:")
    for col, cnt in nan_cols.items():
        print(f"  {col}: {cnt} ({cnt/len(X)*100:.1f}%)")
    # Median imputation (same approach as previous notebooks)
    from sklearn.impute import SimpleImputer
    imputer = SimpleImputer(strategy="median")
    X = pd.DataFrame(imputer.fit_transform(X), columns=FEATURES, index=X.index)
    print(f"  -> Imputed with column medians. NaNs remaining: {X.isna().sum().sum()}")
else:
    print("\nNo missing values.")

# Panel devices
PANEL = {850: "MA-only", 1320: "MA-FA mixed", 119: "FA-Cs"}
print(f"\nPanel devices: {list(PANEL.keys())}")
for idx, label in PANEL.items():
    print(f"  Device {idx} ({label}): T80 = {df.loc[idx, TARGET]:.0f} h")

XGBoost not available -- proceeding with 3 models.
Dataset: 1543 devices

NaN counts per feature:
  Perovskite_band_gap: 474 (30.7%)
  Pb: 3 (0.2%)
  Sn: 3 (0.2%)
  I: 1 (0.1%)
  Br: 1 (0.1%)
  Cl: 1 (0.1%)
  MA: 2 (0.1%)
  FA: 2 (0.1%)
  Cs: 2 (0.1%)
  first_Prvskt_annealing_temperature: 118 (7.6%)
  first_Prvskt_thermal_annealing_time: 139 (9.0%)
  Perovskite_thickness: 1006 (65.2%)
  Cell_area_measured: 41 (2.7%)
  JV_default_Voc: 57 (3.7%)
  JV_default_Jsc: 51 (3.3%)
  JV_default_FF: 58 (3.8%)
  -> Imputed with column medians. NaNs remaining: 0

Panel devices: [850, 1320, 119]
  Device 850 (MA-only): T80 = 3400 h
  Device 1320 (MA-FA mixed): T80 = 940 h
  Device 119 (FA-Cs): T80 = 3423 h


## 2 -- Define Models

In [2]:
def build_models(seed):
    """Return dict of model_name -> fitted-ready model (not yet fitted)."""
    models = {
        "ExtraTrees": ExtraTreesRegressor(
            n_estimators=700, max_features="sqrt",
            min_samples_split=3, min_samples_leaf=1,
            bootstrap=False, random_state=seed,
        ),
        "GradientBoosting": GradientBoostingRegressor(
            n_estimators=500, max_depth=5, learning_rate=0.05,
            random_state=seed,
        ),
        "RandomForest": RandomForestRegressor(
            n_estimators=700, max_features="sqrt",
            random_state=seed,
        ),
    }
    if HAS_XGB:
        models["XGBoost"] = XGBRegressor(
            n_estimators=500, max_depth=5, learning_rate=0.05,
            random_state=seed, verbosity=0,
        )
    return models

model_names = list(build_models(0).keys())
print(f"Models to evaluate ({len(model_names)}): {model_names}")

Models to evaluate (3): ['ExtraTrees', 'GradientBoosting', 'RandomForest']


## 3 -- 20-Split Evaluation Loop

For each split (seeds 0-19):
- 80/20 train/test split
- Train each model on training set
- Rank test-set devices by predicted stability (descending)
- Check if each panel device lands in test set and in top-20

In [3]:
N_SPLITS = 20
TOP_K = 20

# Track: for each model and panel device, count (appearances_in_test, top20_count)
appearances = {m: {d: 0 for d in PANEL} for m in model_names}
top20_hits  = {m: {d: 0 for d in PANEL} for m in model_names}

# Also store full test-set rankings for rank correlation
all_rankings = {m: [] for m in model_names}  # list of (split, ranking_series)

for seed in range(N_SPLITS):
    idx_train, idx_test = train_test_split(
        np.arange(len(X)), test_size=0.2, random_state=seed
    )
    X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
    y_train, y_test = y[idx_train], y[idx_test]

    models = build_models(seed)
    for mname, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        # Build ranking (descending predicted stability)
        rank_series = pd.Series(preds, index=idx_test)
        rank_order = rank_series.sort_values(ascending=False)
        top20_indices = set(rank_order.index[:TOP_K])

        all_rankings[mname].append(rank_series)

        for dev_idx in PANEL:
            if dev_idx in idx_test:
                appearances[mname][dev_idx] += 1
                if dev_idx in top20_indices:
                    top20_hits[mname][dev_idx] += 1

    if (seed + 1) % 5 == 0:
        print(f"  Completed split {seed + 1}/{N_SPLITS}")

print("Done.")

  Completed split 5/20


  Completed split 10/20


  Completed split 15/20


  Completed split 20/20
Done.


## 4 -- Consensus Table: Top-20 Rate per Model and Panel Device

In [4]:
# Build top-20 rate table
rows = []
for dev_idx, label in PANEL.items():
    row = {"Device": dev_idx, "Label": label}
    for mname in model_names:
        app = appearances[mname][dev_idx]
        hits = top20_hits[mname][dev_idx]
        rate = hits / app if app > 0 else np.nan
        row[f"{mname}_appearances"] = app
        row[f"{mname}_top20"] = hits
        row[f"{mname}_top20_rate"] = rate
    rows.append(row)

consensus_df = pd.DataFrame(rows)

# Display a clean summary
print("=" * 80)
print("TOP-20 RATE PER MODEL AND PANEL DEVICE")
print("=" * 80)
for _, r in consensus_df.iterrows():
    print(f"\nDevice {int(r['Device'])} ({r['Label']}):")
    for mname in model_names:
        app = int(r[f"{mname}_appearances"])
        hits = int(r[f"{mname}_top20"])
        rate = r[f"{mname}_top20_rate"]
        print(f"  {mname:20s}: {hits:2d}/{app:2d} = {rate:.1%}")

consensus_df

TOP-20 RATE PER MODEL AND PANEL DEVICE

Device 850 (MA-only):
  ExtraTrees          :  4/ 4 = 100.0%
  GradientBoosting    :  4/ 4 = 100.0%
  RandomForest        :  4/ 4 = 100.0%

Device 1320 (MA-FA mixed):
  ExtraTrees          :  3/ 3 = 100.0%
  GradientBoosting    :  2/ 3 = 66.7%
  RandomForest        :  3/ 3 = 100.0%

Device 119 (FA-Cs):
  ExtraTrees          :  4/ 4 = 100.0%
  GradientBoosting    :  2/ 4 = 50.0%
  RandomForest        :  3/ 4 = 75.0%


,Device,Label,ExtraTrees_appearances,ExtraTrees_top20,ExtraTrees_top20_rate,GradientBoosting_appearances,GradientBoosting_top20,GradientBoosting_top20_rate,RandomForest_appearances,RandomForest_top20,RandomForest_top20_rate
0,850,MA-only,4,4,1.0,4,4,1.000000,4,4,1.00
1,1320,MA-FA mixed,3,3,1.0,3,2,0.666667,3,3,1.00
2,119,FA-Cs,4,4,1.0,4,2,0.500000,4,3,0.75


## 5 -- Rank Correlation (Kendall Tau) Between Model Pairs

In [5]:
# Compute pairwise Kendall tau averaged across splits
from itertools import combinations

tau_results = {}
for m1, m2 in combinations(model_names, 2):
    taus = []
    for split_idx in range(N_SPLITS):
        s1 = all_rankings[m1][split_idx]
        s2 = all_rankings[m2][split_idx]
        # Align on common test indices
        common = s1.index.intersection(s2.index)
        if len(common) > 1:
            tau, _ = kendalltau(s1.loc[common].values, s2.loc[common].values)
            taus.append(tau)
    mean_tau = np.mean(taus)
    tau_results[(m1, m2)] = mean_tau

print("Pairwise Kendall Tau (averaged over 20 splits):")
print("-" * 50)
for (m1, m2), tau in sorted(tau_results.items()):
    print(f"  {m1:20s} vs {m2:20s}: tau = {tau:.4f}")

# Build correlation matrix
tau_matrix = pd.DataFrame(np.ones((len(model_names), len(model_names))),
                          index=model_names, columns=model_names)
for (m1, m2), tau in tau_results.items():
    tau_matrix.loc[m1, m2] = tau
    tau_matrix.loc[m2, m1] = tau

print("\nCorrelation matrix:")
tau_matrix

Pairwise Kendall Tau (averaged over 20 splits):
--------------------------------------------------
  ExtraTrees           vs GradientBoosting    : tau = 0.5872
  ExtraTrees           vs RandomForest        : tau = 0.8028
  GradientBoosting     vs RandomForest        : tau = 0.6417

Correlation matrix:


,ExtraTrees,GradientBoosting,RandomForest
ExtraTrees,1.000000,0.587205,0.802827
GradientBoosting,0.587205,1.000000,0.641743
RandomForest,0.802827,0.641743,1.000000


## 6 -- Consensus Score

For each panel device: what fraction of models rank it in top-20 in >= 80% of its test-set appearances?

In [6]:
THRESHOLD = 0.80  # top-20 rate threshold for "agreement"
n_models = len(model_names)

print("CONSENSUS SCORE (models with top-20 rate >= 80%)")
print("=" * 60)

consensus_scores = {}
for dev_idx, label in PANEL.items():
    agreeing = 0
    for mname in model_names:
        rate = consensus_df.loc[consensus_df["Device"] == dev_idx,
                                f"{mname}_top20_rate"].values[0]
        if rate >= THRESHOLD:
            agreeing += 1
    consensus_scores[dev_idx] = agreeing
    print(f"  Device {dev_idx} ({label}): {agreeing}/{n_models} models agree "
          f"({'PASS' if agreeing >= 3 else 'FAIL'})")

# Add consensus to dataframe
consensus_df["consensus_agreeing_models"] = [
    consensus_scores[d] for d in PANEL
]
consensus_df["consensus_fraction"] = consensus_df["consensus_agreeing_models"] / n_models

print(f"\nDevices with consensus >= 3/{n_models}: "
      f"{sum(1 for v in consensus_scores.values() if v >= 3)}/3")

CONSENSUS SCORE (models with top-20 rate >= 80%)
  Device 850 (MA-only): 3/3 models agree (PASS)
  Device 1320 (MA-FA mixed): 2/3 models agree (FAIL)
  Device 119 (FA-Cs): 1/3 models agree (FAIL)

Devices with consensus >= 3/3: 1/3


## 7 -- Save Results

In [7]:
out_path = "Packet_P011_Multi_Model_Consensus.csv"
consensus_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
consensus_df

Saved: Packet_P011_Multi_Model_Consensus.csv


,Device,Label,ExtraTrees_appearances,ExtraTrees_top20,ExtraTrees_top20_rate,GradientBoosting_appearances,GradientBoosting_top20,GradientBoosting_top20_rate,RandomForest_appearances,RandomForest_top20,RandomForest_top20_rate,consensus_agreeing_models,consensus_fraction
0,850,MA-only,4,4,1.0,4,4,1.000000,4,4,1.00,3,1.000000
1,1320,MA-FA mixed,3,3,1.0,3,2,0.666667,3,3,1.00,2,0.666667
2,119,FA-Cs,4,4,1.0,4,2,0.500000,4,3,0.75,1,0.333333


## 8 -- Honest Status Footer

In [8]:
# Determine status
devices_passing = sum(1 for v in consensus_scores.values() if v >= 3)

if devices_passing == 3:
    status = "Confirmed"
    status_detail = "All 3 panel devices have multi-model consensus (>= 3 models agree on top-20)."
elif devices_passing >= 2:
    status = "Promising"
    status_detail = f"{devices_passing}/3 panel devices have multi-model consensus."
else:
    status = "Negative"
    status_detail = f"Only {devices_passing}/3 panel devices have multi-model consensus."

print("=" * 80)
print("PACKET P-011 -- HONEST STATUS FOOTER")
print("=" * 80)
print(f"\nAgent OS v3.0 Status: ** {status} **")
print(f"\n{status_detail}")
print(f"\nModels evaluated: {n_models} ({', '.join(model_names)})")
print(f"Splits: {N_SPLITS} (seeds 0-{N_SPLITS-1})")
print(f"Ranking criterion: test-set top-{TOP_K} rate >= {THRESHOLD:.0%}")
print()
for dev_idx, label in PANEL.items():
    n_agree = consensus_scores[dev_idx]
    print(f"  Device {dev_idx} ({label}): {n_agree}/{n_models} models agree -> "
          f"{'PASS' if n_agree >= 3 else 'FAIL'}")
print()
print(f"Mean pairwise Kendall tau: {np.mean(list(tau_results.values())):.4f}")
print(f"Output: Packet_P011_Multi_Model_Consensus.csv")
print("=" * 80)

PACKET P-011 -- HONEST STATUS FOOTER

Agent OS v3.0 Status: ** Negative **

Only 1/3 panel devices have multi-model consensus.

Models evaluated: 3 (ExtraTrees, GradientBoosting, RandomForest)
Splits: 20 (seeds 0-19)
Ranking criterion: test-set top-20 rate >= 80%

  Device 850 (MA-only): 3/3 models agree -> PASS
  Device 1320 (MA-FA mixed): 2/3 models agree -> FAIL
  Device 119 (FA-Cs): 1/3 models agree -> FAIL

Mean pairwise Kendall tau: 0.6773
Output: Packet_P011_Multi_Model_Consensus.csv
